In [1]:
import pandas as pd
import numpy as np

# ==========================================
# 1. PASO PREVIO: GENERAR ARCHIVOS FUENTE
# ==========================================
# Maestro de productos (Excel)
df_productos_raw = pd.DataFrame({
    'id_producto': [101, 102, 103, 104, 105],
    'nombre_producto': [' Teclado Mecanico ', 'Mouse Inalambrico', 'Monitor 24', 'Auriculares Gamer', ' Pad Mouse XL '],
    'precio_unitario': [150000.0, 85000.0, 1200000.0, 250000.0, 45000.0]
})
df_productos_raw.to_excel('productos.xlsx', index=False)

# Transacciones de ventas (CSV)
df_ventas_raw = pd.DataFrame({
    'id_venta': [1, 2, 3, 4, 5, 6, 7],
    'id_producto': ['101', '102', '103', '104', '101', '105', '102'],  # Formato texto para simular error común
    'cantidad': [2, 1, np.nan, 3, 1, 4, 2],  # Incluye un nulo
    'fecha': ['2026-09-01', '2026-09-01', '2026-09-02', '2026-09-02', '2026-09-03', '2026-09-03', '2026-09-04']
})
df_ventas_raw.to_csv('ventas.csv', index=False)


# ==========================================
# 2. ADQUISICIÓN DE DATOS (CSV Y EXCEL)
# ==========================================
print("Cargando archivos fuente...")
df_ventas = pd.read_csv('ventas.csv')
df_productos = pd.read_excel('productos.xlsx')


# ==========================================
# 3. TRANSFORMACIÓN Y LIMPIEZA
# ==========================================
# A) Asegurar coherencia de tipos de datos para la llave primaria (ID)
df_ventas['id_producto'] = df_ventas['id_producto'].astype(int)
df_productos['id_producto'] = df_productos['id_producto'].astype(int)

# B) Convertir fecha a objeto datetime
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])

# C) Manejo de valores nulos (Imputación con la mediana o valor por defecto)
df_ventas['cantidad'] = df_ventas['cantidad'].fillna(1).astype(int)

# D) Limpieza de strings
df_productos['nombre_producto'] = df_productos['nombre_producto'].str.strip()

# E) Merge / Unión entre ambas fuentes
df_consolidado = pd.merge(df_ventas, df_productos, on='id_producto', how='inner')


# ==========================================
# 4. NORMALIZACIÓN Y COLUMNA CALCULADA
# ==========================================
# Columna calculada: total_venta
df_consolidado['total_venta'] = df_consolidado['cantidad'] * df_consolidado['precio_unitario']


# ==========================================
# 5. EXPORTACIÓN A PARQUET
# ==========================================
df_consolidado.to_parquet('ventas_consolidadas.parquet', index=False)

print("\n¡Proceso completado con éxito!")
print("\nVista previa del dataset final:")
print(df_consolidado.info())
print("\nPrimeros registros:")
print(df_consolidado.head())



Cargando archivos fuente...

¡Proceso completado con éxito!

Vista previa del dataset final:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_venta         7 non-null      int64         
 1   id_producto      7 non-null      int64         
 2   cantidad         7 non-null      int64         
 3   fecha            7 non-null      datetime64[ns]
 4   nombre_producto  7 non-null      object        
 5   precio_unitario  7 non-null      int64         
 6   total_venta      7 non-null      int64         
dtypes: datetime64[ns](1), int64(5), object(1)
memory usage: 524.0+ bytes
None

Primeros registros:
   id_venta  id_producto  cantidad      fecha    nombre_producto  \
0         1          101         2 2026-09-01   Teclado Mecanico   
1         2          102         1 2026-09-01  Mouse Inalambrico   
2         3          103  